# CARGA DE CSV

In [ ]:
# ==========================================================
# MÓDULO 1 - Carga robusta CSV + guardar con nombre estándar (CORREGIDO)
# ==========================================================

import pandas as pd
from google.colab import files

def leer_csv_robusto(file_name):
    # Encodings típicos de archivos AFIP/Excel/Windows
    encodings = ["utf-8", "utf-8-sig", "latin-1", "cp1252", "utf-16"]
    # Separadores posibles (AFIP suele venir con ;)
    seps = [None, ";", ",", "\t", "|"]

    last_err = None
    for enc in encodings:
        for sep in seps:
            try:
                df = pd.read_csv(
                    file_name,
                    dtype=str,
                    sep=sep,
                    # autodetección de sep requiere python; si fijamos sep, usamos C (más estable/rápido)
                    engine="python" if sep is None else "c",
                    encoding=enc
                )
                print(f"✅ Archivo leído con encoding={enc} | sep={repr(sep)} | columnas={df.shape[1]}")
                return df
            except Exception as e:
                last_err = e
                continue

    raise ValueError(f"❌ No se pudo leer el CSV. Último error: {last_err}")

def cargar_csv_y_preparar_y_guardar():
    print("📤 Cargá el archivo CSV...")
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("No se cargó ningún archivo.")

    file_name = next(iter(uploaded))
    print(f"✅ Archivo cargado: {file_name}")

    df = leer_csv_robusto(file_name).fillna("")

    n_cols = df.shape[1]
    if n_cols == 31:
        tipo = "Comprobantes de importación"
        out_name = "Comprobantes_importacion.csv"
    elif n_cols == 32:
        tipo = "Comprobantes de compras"
        out_name = "Comprobantes_compras.csv"
    else:
        raise ValueError(f"❌ El archivo tiene {n_cols} columnas. Solo se aceptan 31 o 32.")

    print(f"📌 Tipo detectado: {tipo} ({n_cols} columnas)")

    nuevas_cols = [" CUIT emisor/corredor", "Denominación del emisor/corredor", "IVA comisión"]
    for col in nuevas_cols:
        if col not in df.columns:
            df[col] = ""

    print(f"➕ Columnas agregadas. Total ahora: {df.shape[1]} columnas")

    # Guardar con nombre fijo (sin índice) en UTF-8
    df.to_csv(out_name, index=False, encoding="utf-8")
    print(f"💾 Guardado como: {out_name}")

    return df, tipo, out_name

# Ejecutar módulo 1
df, tipo_archivo, archivo_estandar = cargar_csv_y_preparar_y_guardar()
df.head()

📤 Cargá el archivo CSV...


Saving comprobantes_periodo_202602_compras_20260224_1753 (montos expresados en pesos).csv to comprobantes_periodo_202602_compras_20260224_1753 (montos expresados en pesos) (1).csv
✅ Archivo cargado: comprobantes_periodo_202602_compras_20260224_1753 (montos expresados en pesos) (1).csv
✅ Archivo leído con encoding=latin-1 | sep=';' | columnas=32
📌 Tipo detectado: Comprobantes de compras (32 columnas)
➕ Columnas agregadas. Total ahora: 35 columnas
💾 Guardado como: Comprobantes_compras.csv


,Fecha de Emisión,Tipo de Comprobante,Punto de Venta,Número de Comprobante,Tipo Doc. Vendedor,Nro. Doc. Vendedor,Denominación Vendedor,Importe Total,Moneda Original,Tipo de Cambio,...,"Importe IVA 10,5%",Neto Gravado IVA 21%,Importe IVA 21%,Neto Gravado IVA 27%,Importe IVA 27%,Total Neto Gravado,Total IVA,CUIT emisor/corredor,Denominación del emisor/corredor,IVA comisión
0,1/2/2026,11,3,6564,80,20344538485,KEINER EMILIANO ANTONIO,9398,PES,1,...,0,0,0,0,0,0,0,,,
1,1/2/2026,81,15,11599,80,30709658006,CARTUJA S A,"69999,96",PES,1,...,0,"54993,33","11548,6",0,0,"54993,33","11548,6",,,
2,2/2/2026,1,1,1162,80,20128290959,MAGALLANES MARIO ALBERTO,"415250,22",PES,1,...,,343182,"72068,22",,,343182,"72068,2",,,
3,17/2/2026,1,6,108956,80,33711063299,COMERCIAL RANQUELES S.R.L.,"98552,8",PES,1,...,,"71631,17","15042,54",,,"71631,17","15042,5",,,
4,2/2/2026,1,45,6729,80,30683958073,AGROEMPRESA SAN FRANCISCO S A,136257,PES,1,...,"12947,5",,,,,"123309,5","12947,5",,,


# COMPROBANTES COMPRAS TXT + TXT ALICUOTAS


In [ ]:
# ==========================================================
# MÓDULO 2 - Compras: generar línea TXT (325 chars) por fila
# ==========================================================

import os
import pandas as pd

def leer_csv_robusto_archivo_local(path):
    encodings = ["utf-8", "latin-1", "cp1252"]
    for enc in encodings:
        try:
            df = pd.read_csv(path, dtype=str, sep=None, engine="python", encoding=enc).fillna("")
            print(f"✅ Leído {path} con encoding: {enc}")
            return df
        except Exception:
            continue
    raise ValueError(f"❌ No pude leer {path} con encodings habituales.")

def cargar_df_compras_desde_estandar():
    path = "Comprobantes_compras.csv"
    if not os.path.exists(path):
        raise FileNotFoundError("❌ No existe Comprobantes_compras.csv. Ejecutá el Módulo 1 cargando un CSV de compras (32 columnas).")
    dfc = leer_csv_robusto_archivo_local(path)
    if dfc.shape[1] != 35:
        raise ValueError(f"❌ Se esperaba 35 columnas para compras (32+3). Tiene {dfc.shape[1]}.")
    return dfc

# ---------- helpers por posición ----------
def col_by_letter(letter: str) -> int:
    # A->0, B->1 ... Z->25, AA->26, AB->27 ...
    letter = letter.strip().upper()
    n = 0
    for ch in letter:
        n = n * 26 + (ord(ch) - ord('A') + 1)
    return n - 1

def v(row, letter):
    return row.iloc[col_by_letter(letter)]

#def format_fecha_dmy_a_aaaammdd(val):
    # val: d/m/yyyy (o dd/mm/yyyy)
   # dt = pd.to_datetime(str(val), dayfirst=True, errors="coerce")
   # if pd.isna(dt):
   #     return "0"*8
  #  return dt.strftime("%Y%m%d")

import re
from datetime import date

def format_fecha_dmy_a_aaaammdd(val):
    s = "" if val is None else str(val).strip()

    if not s:
        return "0" * 8

    # Caso ISO: 2026-02-01 (o con hora)
    m = re.match(r"^(\d{4})-(\d{2})-(\d{2})", s)
    if m:
        y, mm, dd = m.group(1), m.group(2), m.group(3)
        return f"{y}{mm}{dd}"

    # Caso D/M/YYYY o DD/MM/YYYY
    if "/" in s:
        parts = s.split("/")
        if len(parts) >= 3:
            try:
                d = int(parts[0])
                mth = int(parts[1])
                y = int(parts[2])

                # Si tuvieras algún archivo “al revés” (M/D) y querés corregirlo automáticamente:
                # si el "mes" es >12 y el "día" <=12, intercambiamos
                if mth > 12 and d <= 12:
                    d, mth = mth, d

                return date(y, mth, d).strftime("%Y%m%d")
            except:
                return "0" * 8

    # Fallback: intentá pandas, pero como último recurso
    dt = pd.to_datetime(s, dayfirst=True, errors="coerce")
    if pd.isna(dt):
        return "0" * 8
    return dt.strftime("%Y%m%d")

def pad_left_zeros(val, length):
    s = "" if val is None else str(val)
    return s.zfill(length)[:length]  # si viene más largo, corta a la izquierda (seguro para numéricos)

def pad_right_text(val, length):
    s = "" if val is None else str(val)
    s = s[:length]
    return s.ljust(length)

def parse_float(val):
    s = "" if val is None else str(val).strip()
    if s == "":
        return 0.0
    # soportar 1.234,56 y 1234,56 y 1234.56
    s = s.replace(" ", "")
    if "," in s and "." in s:
        s = s.replace(".", "").replace(",", ".")
    else:
        s = s.replace(",", ".")
    try:
        return float(s)
    except:
        return 0.0

def importe_15(val):
    x = round(parse_float(val), 2)
    return str(int(round(x * 100))).zfill(15)

def tipo_cambio_10(val):
    x = round(parse_float(val), 6)
    return str(int(round(x * 1_000_000))).zfill(10)

def es_distinto_de_cero(val):
    x = parse_float(val)
    return abs(x) > 0.0

def contar_alicuotas(row):
    grupos = {
        "0%":   ["T"],
        "2.5%": ["U", "V"],
        "5%":   ["W", "X"],
        "10.5%":["Y", "Z"],
        "21%":  ["AA", "AB"],
        "27%":  ["AC", "AD"],
    }
    c = 0
    for cols in grupos.values():
        if any(es_distinto_de_cero(v(row, col)) for col in cols):
            c += 1
    return str(c)  # 1 caracter

def campo20_importacion(b_val):
    # Si B == 11 -> "E" sino espacio
    b = str(b_val).strip()
    # por si viene "011"
    try:
        b_num = int(b)
    except:
        b_num = None
    return "E" if b_num == 11 else " "

def generar_linea_compras(row):
    campos = []
    # 1 a 9
    campos.append(format_fecha_dmy_a_aaaammdd(v(row, "A")))    # 8
    campos.append(pad_left_zeros(v(row, "B"), 3))              # 3
    campos.append(pad_left_zeros(v(row, "C"), 5))              # 5
    campos.append(pad_left_zeros(v(row, "D"), 20))             # 20
    campos.append("0"*16)                                      # 16 (campo 5)
    campos.append(pad_left_zeros(v(row, "E"), 2))              # 2
    campos.append(pad_left_zeros(v(row, "F"), 20))             # 20
    campos.append(pad_right_text(v(row, "G"), 30))             # 30
    campos.append(importe_15(v(row, "H")))                      # 15
    # 10 a 16 (15 cada uno)
    campos.append(importe_15(v(row, "K")))  # 10
    campos.append(importe_15(v(row, "L")))  # 11
    campos.append(importe_15(v(row, "Q")))  # 12
    campos.append(importe_15(v(row, "N")))  # 13
    campos.append(importe_15(v(row, "O")))  # 14
    campos.append(importe_15(v(row, "P")))  # 15
    campos.append(importe_15(v(row, "R")))  # 16
    # 17 y 18
    campos.append("PES")
    campos.append(tipo_cambio_10(1))   # devuelve "0001000000"
    # 19 y 20
    campos.append(contar_alicuotas(row))                       # 19 (1)
    campos.append(campo20_importacion(v(row, "B")))                                         # 20 (1) por ahora
    # 21 y 22
    campos.append(importe_15(v(row, "M")))                      # 21 (15)
    campos.append(importe_15(v(row, "S")))                      # 22 (15)
    # 23, 24, 25 (AG, AH, AI son las 3 nuevas columnas al final)
    campos.append(pad_left_zeros(v(row, "AG"), 11))            # 23 (11)
    campos.append(pad_right_text(v(row, "AH"), 30))            # 24 (30)
    campos.append(importe_15(v(row, "AI")))                    # 25 (15)

    linea = "".join(campos)

    if len(linea) != 325:
        raise ValueError(f"❌ Línea con {len(linea)} caracteres. Debe ser 325.")
    return linea

# ---- cargar compras y probar 1 fila ----
df_compras = cargar_df_compras_desde_estandar()

linea_test = generar_linea_compras(df_compras.iloc[0])
print("✅ Longitud:", len(linea_test))
print(linea_test[:200])

# ==========================================================
# EXTENSIÓN MÓDULO 2 (COMPRAS) - TXT ALICUOTAS (84 chars)
# Archivo: LIBRO_IVA_DIGITAL_COMPRAS_ALICUOTAS
# ==========================================================

# Mapeo según tu especificación:
# Neto gravado:
# 0%  -> T
# 2.5 -> U
# 5%  -> W
# 10.5-> Y
# 21% -> AA
# 27% -> AC
#
# Impuesto liquidado:
# 0%  -> 15 ceros
# 2.5 -> V
# 5%  -> X
# 10.5-> Z
# 21% -> AB
# 27% -> AD
#
# Código alícuota:
# 0%   -> 0003
# 10.5 -> 0004
# 21%  -> 0005
# 27%  -> 0006
# 5%   -> 0008
# 2.5  -> 0009

ALICUOTAS_COMPRAS = [
    {"rate": "0",    "neto_col": "T",  "iva_col": None, "codigo": "0003"},
    {"rate": "2.5",  "neto_col": "U",  "iva_col": "V", "codigo": "0009"},
    {"rate": "5",    "neto_col": "W",  "iva_col": "X", "codigo": "0008"},
    {"rate": "10.5", "neto_col": "Y",  "iva_col": "Z", "codigo": "0004"},
    {"rate": "21",   "neto_col": "AA", "iva_col": "AB","codigo": "0005"},
    {"rate": "27",   "neto_col": "AC", "iva_col": "AD","codigo": "0006"},
]

def hay_movimiento_alicuota_compras(row, spec):
    neto = v(row, spec["neto_col"])
    iva = "0" if spec["iva_col"] is None else v(row, spec["iva_col"])
    return es_distinto_de_cero(neto) or es_distinto_de_cero(iva)

def generar_linea_alicuota_compras(row, spec):
    # Campos 1..5 (comunes)
    c1 = pad_left_zeros(v(row, "B"), 3)    # Tipo comprobante
    c2 = pad_left_zeros(v(row, "C"), 5)    # Punto de venta
    c3 = pad_left_zeros(v(row, "D"), 20)   # Número comprobante
    c4 = pad_left_zeros(v(row, "E"), 2)    # Código doc vendedor
    c5 = pad_left_zeros(v(row, "F"), 20)   # Nro identificación vendedor

    # Campo 6 (neto gravado)
    c6 = importe_15(v(row, spec["neto_col"]))  # 15

    # Campo 7 (código alícuota)
    c7 = spec["codigo"]  # 4

    # Campo 8 (impuesto liquidado)
    if spec["iva_col"] is None:
        c8 = "0" * 15
    else:
        c8 = importe_15(v(row, spec["iva_col"]))

    linea = f"{c1}{c2}{c3}{c4}{c5}{c6}{c7}{c8}"

    if len(linea) != 84:
        raise ValueError(f"❌ Línea ALICUOTA con {len(linea)} caracteres. Debe ser 84.")
    return linea

def generar_lineas_alicuotas_compras(row):
    # Genera 0..N líneas por comprobante (según alícuotas con movimiento)
    lineas = []
    for spec in ALICUOTAS_COMPRAS:
        if hay_movimiento_alicuota_compras(row, spec):
            lineas.append(generar_linea_alicuota_compras(row, spec))
    return lineas

# ---- PRUEBA: comparar campo 19 vs cantidad de líneas generadas ----
row0 = df_compras.iloc[0]
lineas0 = generar_lineas_alicuotas_compras(row0)

print("Campo 19 (según tu función):", contar_alicuotas(row0))
print("Líneas ALICUOTAS generadas:", len(lineas0))
if lineas0:
    print("Ejemplo línea (84):", len(lineas0[0]), lineas0[0])

# ---- EXPORTAR TXT ALICUOTAS COMPRAS ----
from google.colab import files

def exportar_txt_alicuotas_compras(df_compras, nombre_salida="LIBRO_IVA_DIGITAL_COMPRAS_ALICUOTAS.txt"):
    lineas = []
    errores = []

    for idx in range(len(df_compras)):
        try:
            lineas.extend(generar_lineas_alicuotas_compras(df_compras.iloc[idx]))
        except Exception as e:
            errores.append((idx, str(e)))

    if errores:
        print("❌ Errores (primeros 10):")
        for i, msg in errores[:10]:
            print(f" - Fila {i}: {msg}")
        raise ValueError(f"Hay {len(errores)} filas con error. No se generó el TXT.")

    with open(nombre_salida, "w", encoding="utf-8", newline="\n") as f:
        f.write("\n".join(lineas))

    print(f"✅ TXT ALICUOTAS generado: {nombre_salida}")
    print(f"📌 Registros (líneas): {len(lineas)} | Longitud por línea: {len(lineas[0]) if lineas else 0}")

    files.download(nombre_salida)

✅ Leído Comprobantes_compras.csv con encoding: utf-8
✅ Longitud: 325
20260201011000030000000000000000656400000000000000008000000000020344538485KEINER EMILIANO ANTONIO       000000000939800000000000000000000000000000000000000000000000000000000000000000000000000000000000
Campo 19 (según tu función): 0
Líneas ALICUOTAS generadas: 0


# TXT IMPORTACIÓN + TXT ALICUOTAS


In [ ]:
# ==========================================================
# MÓDULO 2B - Importación: generar línea TXT (325 chars) por fila
# ==========================================================

import os
import pandas as pd

def cargar_df_importacion_desde_estandar():
    path = "Comprobantes_importacion.csv"
    if not os.path.exists(path):
        raise FileNotFoundError("❌ No existe Comprobantes_importacion.csv. Ejecutá el Módulo 1 cargando un CSV de importación (31 columnas).")
    dfc = leer_csv_robusto_archivo_local(path).fillna("")
    if dfc.shape[1] != 34:
        raise ValueError(f"❌ Se esperaba 34 columnas para importación (31+3). Tiene {dfc.shape[1]}.")
    return dfc

# Helpers (si ya los tenés definidos arriba, NO vuelvas a pegarlos)
# col_by_letter, v, format_fecha_dmy_a_aaaammdd, pad_left_zeros, pad_right_text,
# parse_float, importe_15, tipo_cambio_10, es_distinto_de_cero

def contar_alicuotas_importacion(row):
    grupos = {
        "0%":   ["S"],
        "2.5%": ["T", "U"],
        "5%":   ["V", "W"],
        "10.5%":["X", "Y"],
        "21%":  ["Z", "AA"],
        "27%":  ["AB", "AC"],
    }
    c = 0
    for cols in grupos.values():
        if any(es_distinto_de_cero(v(row, col)) for col in cols):
            c += 1
    return str(c)  # 1 caracter

def campo20_importacion(b_val):
    # Si B == 11 -> "E" sino espacio
    b = str(b_val).strip()
    # por si viene "011"
    try:
        b_num = int(b)
    except:
        b_num = None
    return "E" if b_num == 11 else " "

def generar_linea_importacion(row):
    campos = []

    # 1
    campos.append(format_fecha_dmy_a_aaaammdd(v(row, "A")))   # 8

    # 2
    campos.append(pad_left_zeros(v(row, "B"), 3))             # 3

    # 3 fijo
    campos.append("00000")                                    # 5

    # 4 fijo
    campos.append("0"*20)                                     # 20

    # 5 = C (16)
    campos.append(pad_left_zeros(v(row, "C"), 16))            # 16

    # 6 = D (2)
    campos.append(pad_left_zeros(v(row, "D"), 2))             # 2

    # 7 = E (20)
    campos.append(pad_left_zeros(v(row, "E"), 20))            # 20

    # 8 = F (30 texto)
    campos.append(pad_right_text(v(row, "F"), 30))            # 30

    # 9 = G (15 importe)
    campos.append(importe_15(v(row, "G")))                    # 15

    # 10 a 16 (15 cada uno)
    campos.append(importe_15(v(row, "J")))  # 10
    campos.append(importe_15(v(row, "K")))  # 11
    campos.append(importe_15(v(row, "P")))  # 12
    campos.append(importe_15(v(row, "M")))  # 13
    campos.append(importe_15(v(row, "N")))  # 14
    campos.append(importe_15(v(row, "O")))  # 15
    campos.append(importe_15(v(row, "Q")))  # 16

    # 17 y 18 fijos
    campos.append("PES")                   # 17 (3)
    campos.append(tipo_cambio_10(1))       # 18 (10) => 0001000000

    # 19 alícuotas
    campos.append(contar_alicuotas_importacion(row))          # 19 (1)

    # 20 depende de B
    campos.append(campo20_importacion(v(row, "B")))           # 20 (1)

    # 21 y 22
    campos.append(importe_15(v(row, "L")))                    # 21 (15)
    campos.append(importe_15(v(row, "R")))                    # 22 (15)

    # 23,24,25 (AF, AG, AH) — son las 3 nuevas columnas al final del DF de importación
    campos.append(pad_left_zeros(v(row, "AF"), 11))           # 23 (11)
    campos.append(pad_right_text(v(row, "AG"), 30))           # 24 (30)
    campos.append(importe_15(v(row, "AH")))                   # 25 (15)

    linea = "".join(campos)

    if len(linea) != 325:
        raise ValueError(f"❌ Línea importación con {len(linea)} caracteres. Debe ser 325.")
    return linea

# ---- cargar importación y probar 1 fila ----
df_import = cargar_df_importacion_desde_estandar()

linea_test_imp = generar_linea_importacion(df_import.iloc[0])
print("✅ Longitud:", len(linea_test_imp))
print(linea_test_imp[:200])

# ===========================================
# MÓDULO 3 - Exportar y Descargar TXT Compras (CBTE + ALICUOTAS)
# ===========================================

from google.colab import files

def exportar_y_descargar_txt_compras(df_compras,
                                    nombre_cbte="LIBRO_IVA_DIGITAL_COMPRAS_CBTE.txt",
                                    nombre_alicuotas="LIBRO_IVA_DIGITAL_COMPRAS_ALICUOTAS.txt"):
    # -------- CBTE --------
    lineas_cbte, errores_cbte = [], []
    for idx in range(len(df_compras)):
        try:
            lineas_cbte.append(generar_linea_compras(df_compras.iloc[idx]))
        except Exception as e:
            errores_cbte.append((idx, str(e)))

    if errores_cbte:
        print("❌ Errores CBTE (primeros 10):")
        for i, msg in errores_cbte[:10]:
            print(f" - Fila {i}: {msg}")
        raise ValueError(f"Hay {len(errores_cbte)} filas con error. No se generó {nombre_cbte}.")

    with open(nombre_cbte, "w", encoding="utf-8", newline="\n") as f:
        f.write("\n".join(lineas_cbte))

    print(f"✅ Generado: {nombre_cbte} | líneas: {len(lineas_cbte)} | largo: {len(lineas_cbte[0]) if lineas_cbte else 0}")

    # -------- ALICUOTAS --------
    lineas_aliq, errores_aliq = [], []
    for idx in range(len(df_compras)):
        try:
            lineas_aliq.extend(generar_lineas_alicuotas_compras(df_compras.iloc[idx]))
        except Exception as e:
            errores_aliq.append((idx, str(e)))

    if errores_aliq:
        print("❌ Errores ALICUOTAS (primeros 10):")
        for i, msg in errores_aliq[:10]:
            print(f" - Fila {i}: {msg}")
        raise ValueError(f"Hay {len(errores_aliq)} filas con error. No se generó {nombre_alicuotas}.")

    with open(nombre_alicuotas, "w", encoding="utf-8", newline="\n") as f:
        f.write("\n".join(lineas_aliq))

    print(f"✅ Generado: {nombre_alicuotas} | líneas: {len(lineas_aliq)} | largo: {len(lineas_aliq[0]) if lineas_aliq else 0}")

    # -------- DESCARGAS (solo acá) --------
    files.download(nombre_cbte)
    files.download(nombre_alicuotas)

    print("📥 Descarga solicitada (si el navegador bloquea, permitir descargas).")


# Ejecutar módulo 3 (esto dispara la generación + descarga)
exportar_y_descargar_txt_compras(df_compras)

FileNotFoundError: ❌ No existe Comprobantes_importacion.csv. Ejecutá el Módulo 1 cargando un CSV de importación (31 columnas).

# EXPORTACIÓN TXT

In [ ]:
# ===========================================
# MÓDULO 3 - Exportador autónomo (detecta archivos + ZIP)
# ===========================================

from google.colab import files
import os, zipfile
import pandas as pd

def leer_csv_robusto_archivo_local(path):
    encodings = ["utf-8", "utf-8-sig", "latin-1", "cp1252", "utf-16"]
    seps = [None, ";", ",", "\t", "|"]
    last_err = None

    for enc in encodings:
        for sep in seps:
            try:
                df = pd.read_csv(
                    path,
                    dtype=str,
                    sep=sep,
                    engine="python" if sep is None else "c",
                    encoding=enc
                ).fillna("")
                print(f"✅ Leído {path} | encoding={enc} | sep={repr(sep)} | cols={df.shape[1]}")
                return df
            except Exception as e:
                last_err = e
                continue
    raise ValueError(f"❌ No pude leer {path}. Último error: {last_err}")

def _exportar_txt_desde_lineas(lineas, nombre_salida):
    with open(nombre_salida, "w", encoding="utf-8", newline="\n") as f:
        f.write("\n".join(lineas))
    print(f"✅ Generado: {nombre_salida} | líneas: {len(lineas)} | largo: {len(lineas[0]) if lineas else 0}")

def _armar_cbte(df, fn_generar_linea, etiqueta):
    lineas, errores = [], []
    for idx in range(len(df)):
        try:
            lineas.append(fn_generar_linea(df.iloc[idx]))
        except Exception as e:
            errores.append((idx, str(e)))
    if errores:
        print(f"❌ Errores {etiqueta} (primeros 10):")
        for i, msg in errores[:10]:
            print(f" - Fila {i}: {msg}")
        raise ValueError(f"Hay {len(errores)} filas con error en {etiqueta}.")
    return lineas

def _armar_alicuotas(df, fn_generar_lineas, etiqueta):
    lineas, errores = [], []
    for idx in range(len(df)):
        try:
            lineas.extend(fn_generar_lineas(df.iloc[idx]))
        except Exception as e:
            errores.append((idx, str(e)))
    if errores:
        print(f"❌ Errores {etiqueta} (primeros 10):")
        for i, msg in errores[:10]:
            print(f" - Fila {i}: {msg}")
        raise ValueError(f"Hay {len(errores)} filas con error en {etiqueta}.")
    return lineas

def exportar_y_descargar_zip_autodetect(
    zip_name="LIBRO_IVA_DIGITAL_SALIDA.zip",
    nombre_compras_cbte="LIBRO_IVA_DIGITAL_COMPRAS_CBTE.txt",
    nombre_compras_alicuotas="LIBRO_IVA_DIGITAL_COMPRAS_ALICUOTAS.txt",
    nombre_import_cbte="LIBRO_IVA_DIGITAL_IMPORTACION_CBTE.txt",
    nombre_import_alicuotas="LIBRO_IVA_DIGITAL_IMPORTACION_BIENES_ALICUOTA.txt",
):
    archivos = []

    # -------------------------
    # Detectar y cargar COMPRAS
    # -------------------------
    if os.path.exists("Comprobantes_compras.csv"):
        if "df_compras" not in globals():
            globals()["df_compras"] = leer_csv_robusto_archivo_local("Comprobantes_compras.csv")
            print("ℹ️ df_compras cargado desde archivo.")
        else:
            print("ℹ️ df_compras ya estaba en memoria.")

        # Exportar solo si existen las funciones necesarias
        if "generar_linea_compras" in globals():
            lineas_cbte = _armar_cbte(df_compras, generar_linea_compras, "COMPRAS_CBTE")
            _exportar_txt_desde_lineas(lineas_cbte, nombre_compras_cbte)
            archivos.append(nombre_compras_cbte)
        else:
            print("⚠️ Falta generar_linea_compras(). Salteo COMPRAS_CBTE.")

        if "generar_lineas_alicuotas_compras" in globals():
            lineas_aliq = _armar_alicuotas(df_compras, generar_lineas_alicuotas_compras, "COMPRAS_ALICUOTAS")
            _exportar_txt_desde_lineas(lineas_aliq, nombre_compras_alicuotas)
            archivos.append(nombre_compras_alicuotas)
        else:
            print("⚠️ Falta generar_lineas_alicuotas_compras(). Salteo COMPRAS_ALICUOTAS.")
    else:
        print("ℹ️ No existe Comprobantes_compras.csv. No exporto COMPRAS.")

    # ------------------------------
    # Detectar y cargar IMPORTACIÓN
    # ------------------------------
    if os.path.exists("Comprobantes_importacion.csv"):
        if "df_import" not in globals():
            globals()["df_import"] = leer_csv_robusto_archivo_local("Comprobantes_importacion.csv")
            print("ℹ️ df_import cargado desde archivo.")
        else:
            print("ℹ️ df_import ya estaba en memoria.")

        if "generar_linea_importacion" in globals():
            lineas_cbte_imp = _armar_cbte(df_import, generar_linea_importacion, "IMPORTACION_CBTE")
            _exportar_txt_desde_lineas(lineas_cbte_imp, nombre_import_cbte)
            archivos.append(nombre_import_cbte)
        else:
            print("⚠️ Falta generar_linea_importacion(). Salteo IMPORTACION_CBTE.")

        if "generar_lineas_alicuotas_import" in globals():
            lineas_aliq_imp = _armar_alicuotas(df_import, generar_lineas_alicuotas_import, "IMPORTACION_ALICUOTAS")
            _exportar_txt_desde_lineas(lineas_aliq_imp, nombre_import_alicuotas)
            archivos.append(nombre_import_alicuotas)
        else:
            print("⚠️ Falta generar_lineas_alicuotas_import(). Salteo IMPORTACION_ALICUOTAS.")
    else:
        print("ℹ️ No existe Comprobantes_importacion.csv. No exporto IMPORTACIÓN.")

    if not archivos:
        raise ValueError("❌ No se generó ningún TXT. (¿Faltan módulos 2/2B o no hay archivos estándar?)")

    # -------------------------
    # ZIP + descarga única
    # -------------------------
    if os.path.exists(zip_name):
        os.remove(zip_name)

    with zipfile.ZipFile(zip_name, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for a in archivos:
            z.write(a)

    print(f"📦 ZIP generado: {zip_name} | archivos dentro: {len(archivos)}")
    files.download(zip_name)

# Ejecutar el exportador autónomo (esto genera + descarga el ZIP)
exportar_y_descargar_zip_autodetect()

ℹ️ df_compras ya estaba en memoria.
✅ Generado: LIBRO_IVA_DIGITAL_COMPRAS_CBTE.txt | líneas: 57 | largo: 325
✅ Generado: LIBRO_IVA_DIGITAL_COMPRAS_ALICUOTAS.txt | líneas: 60 | largo: 84
ℹ️ No existe Comprobantes_importacion.csv. No exporto IMPORTACIÓN.
📦 ZIP generado: LIBRO_IVA_DIGITAL_SALIDA.zip | archivos dentro: 2


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>